# Adatacore CoreForge — Hindi AI v0.2 (Qwen3-1.7B)
This is **a new experiment**, not a replacement for your currently deployed Adatacore chatbot. It first tests the untouched base model; only then does it prepare larger Hindi/Hinglish data and fine-tune with LoRA. The results stay in this Colab session until downloaded.

**GPU:** Runtime → Change runtime type → T4 GPU. Free GPU availability is not guaranteed.
**Data:** Public Aya Hindi and Hinglish instruction datasets, with provenance, license manifest, privacy filtering and separate held-out splits. Synthetic Hinglish content requires human quality review. Do not include private client or contributor data.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Choose Runtime → Change runtime type → T4 GPU before proceeding.")
print("GPU detected:", torch.cuda.get_device_name(0))

In [ ]:
!git clone -q https://github.com/ajay333X/my-adatacore-site.git /content/adatacore-v02 || true
%cd /content/adatacore-v02
!pip -q install 'unsloth' 'datasets>=3.0,<5' 'transformers>=4.51,<5' 'accelerate>=1.4,<2' 'peft>=0.15,<1'

## Step 1 — Base-model quality gate
Before spending GPU time training, the unmodified Qwen3-1.7B must be able to answer basic Hindi prompts sensibly. The previous 0.6B checkpoint repeatedly produced nonsense. This section tests generation and strips special chat tokens appropriately. Inspect the output; do not proceed if it is garbled.

In [ ]:
from unsloth import FastLanguageModel
from transformers import GenerationConfig
BASE = "Qwen/Qwen3-1.7B"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE, max_seq_length=1024, load_in_4bit=True
)
FastLanguageModel.for_inference(model)
test_prompts = [
    "WER और CER में क्या अंतर है? हिंदी में दो वाक्यों में बताइए।",
    "क्या ट्रांसक्रिप्शन में ऑडियो सुने बिना बोले गए शब्द बदलने चाहिए? कारण बताइए।",
    "एक विनम्र हिंदी QA फीडबैक लिखिए जिसमें एक वर्तनी की गलती सुधरवानी हो।",
]
def respond(model, question):
    messages=[{"role":"user","content":question}]
    text=tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    encoded=tokenizer(text,return_tensors="pt").to("cuda")
    with torch.inference_mode():
        generated=model.generate(
            **encoded,max_new_tokens=256,do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(generated[0][encoded["input_ids"].shape[1]:],skip_special_tokens=True).strip()
for q in test_prompts:
    print("\nQUESTION:",q,"\nBASE RESPONSE:",respond(model,q),"\n")

## Step 2 — Explicit quality approval
**Read the three answers above.** Change `BASELINE_READABLE` to `True` only if the base model is producing meaningful Hindi. This prevents spending GPU time fine-tuning a broken baseline. This manual check is intentional; the notebook cannot reliably score arbitrary Hindi answer quality by itself.

In [ ]:
BASELINE_READABLE = False   # Set True ONLY after inspecting the generated Hindi answers
if not BASELINE_READABLE:
    raise RuntimeError("Base model not approved. If output is garbled, stop here and share a screenshot.")

## Step 3 — Prepare larger Hindi dataset
Import up to 1,500 examples each from Aya Hindi (human-annotated) and Hinglish Instruct (synthetic). Counts may be lower if source data is exhausted or filtered. The pipeline records manifests and SHA-256 hashes for train, validation and test. This does not train yet.

In [ ]:
!python workers/coreforge_hindi_data.py --sources aya_hindi,hinglish --limit 1500 --accept-license --out /content/coreforge-hindi-v02-data
import json
from pathlib import Path
m=json.loads(Path("/content/coreforge-hindi-v02-data/manifest.json").read_text())
print("Imported:",m["prepared_count"],"Splits:",m["splits"])
print("Remember: synthetic data must be reviewed before production promotion.")

## Step 4 — Release baseline GPU memory
The training script loads its own model and tokenizer with Unsloth. Release the initial baseline model first to minimize T4 memory pressure.

In [ ]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()
print("Baseline model unloaded.")

## Step 5 — Train v0.2 (research)
This runs the existing CoreForge LoRA trainer on Qwen3-1.7B (4-bit) using the new Hindi dataset. It saves a LoRA adapter, validation/test losses and manifest. Do not publish or use for production until evaluated. If you encounter CUDA out-of-memory, stop and share the error rather than repeatedly retrying.

In [ ]:
!python workers/coreforge_chat_train.py --data /content/coreforge-hindi-v02-data --output /content/coreforge-hindi-v02-checkpoint --model Qwen/Qwen3-1.7B --max-tokens 768 --max-steps 200 --approve-research-run

## Step 6 — Verify report and preserve files
If training completed successfully, download the checkpoint ZIP **and** the JSON report. Colab storage is temporary. The ZIP contains the adapter and tokenizer; it does not contain the full base model.

In [ ]:
import json, zipfile
from pathlib import Path
from google.colab import files
root=Path("/content/coreforge-hindi-v02-checkpoint")
report_file=root/"training_report.json"
weights=root/"adapter"/"adapter_model.safetensors"
assert report_file.exists() and weights.exists(), "Checkpoint incomplete; inspect training output."
report=json.loads(report_file.read_text())
print("Base model:",report["base_model"])
print("Test loss:",report["test_metrics"].get("eval_loss"))
zip_path=Path("/content/Adatacore-Hindi-AI-v0.2.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in (root/"adapter").rglob("*"):
        if p.is_file(): z.write(p,p.relative_to(root.parent))
    z.write(report_file,report_file.relative_to(root.parent))
files.download(str(zip_path))
files.download(str(report_file))

### After the notebook
Upload the ZIP and training report into this ChatGPT conversation. Next we compare baseline vs v0.2 on held-out Hindi QA prompts, detect repetition and verify decoding. **Do not replace the live production chatbot before a qualitative and quantitative evaluation.**